# GPU-Accelerated QX Test for Population Genetics (Python Version)

## Overview

This notebook demonstrates a **GPU-accelerated implementation** of the **QX variance test** for detecting differences in polygenic selection between populations, rewritten in Python for speed and flexibility.


**Reference:**  
Joshi PK et al. (2015) *Directional dominance on stature and cognition in diverse human populations.*  
Nature. [doi:10.1038/nature14618](https://doi.org/10.1038/nature14618)


**Note:** This notebook uses **simulated data** to demonstrate the GPU optimization technique. Real genomic data is not included.


## 1. Setup & Dependencies

Install and import required Python libraries. This notebook uses `cupy` for GPU acceleration and `numpy` for CPU operations.

In [1]:
# Install required packages if not already installed
# !pip install cupy numpy

import cupy as cp
import numpy as np

print(f"✅ CuPy version: {cp.__version__}")
print(f"✅ GPU available: {cp.cuda.is_available()}")
if cp.cuda.is_available():
    device_count = cp.cuda.runtime.getDeviceCount()
    print(f"✅ GPU devices found: {device_count}")
    device_id = cp.cuda.Device().id
    print(f"✅ Using GPU device: {device_id}")

✅ CuPy version: 13.6.0
✅ GPU available: True
✅ GPU devices found: 1
✅ Using GPU device: 0


## 2. Generate Simulated Data

We will create realistic simulated genetic data mimicking:
- SNP effect sizes (beta coefficients from GWAS)
- Population allele frequencies (MAFs) for two populations
- Population structure (different MAF distributions)
- Missing data (~2% NAs, realistic for GWAS quality control)

**NA Handling:**
- Remove entire SNPs (rows) that have NA in either MAF or beta
- Ensures observed and permuted calculations use identical SNP sets
- Keep `np.nan` handling in sums as a failsafe (shouldn't trigger after pre-filtering)


## 3. QX Statistic: CPU Baseline Implementation

This function computes the QX statistic and empirical p-value using CPU (NumPy). It serves as a reference for validating GPU results.

In [4]:
def fst(maf1, maf2, n1=None, n2=None):
    """Compute Hudson's Fst (simplified for synthetic data)."""
    if n1 is not None and n2 is not None:
        num = (maf1 - maf2) ** 2 - maf1 * (1 - maf1) / n1 - maf2 * (1 - maf2) / n2
        denom = maf1 * (1 - maf2) + maf2 * (1 - maf1) + 1e-12
        fst_vals = np.where(denom > 0, num / denom, 0)
    else:
        fst_num = (maf1 - maf2) ** 2
        fst_denom = maf1 * (1 - maf2) + maf2 * (1 - maf1) + 1e-12
        fst_vals = np.where(fst_denom > 0, fst_num / fst_denom, 0)
    return fst_vals

def compute_qx_cpu(maf1, maf2, beta, n_perm=10000, seed=None, n1=None, n2=None, neutral_maf1=None, neutral_maf2=None):
    if seed is not None:
        np.random.seed(seed)
    # Remove rows with any NA
    mask = ~np.isnan(maf1) & ~np.isnan(maf2) & ~np.isnan(beta)
    maf1 = maf1[mask]
    maf2 = maf2[mask]
    beta = beta[mask]
    if len(beta) == 0:
        return {"Qx": np.nan, "Fst": np.nan, "p_value": np.nan}
    diff_obs = maf1 * beta - maf2 * beta
    N_Qx_obs = np.sum(diff_obs) ** 2
    if neutral_maf1 is not None and neutral_maf2 is not None:
        keep_neut = ~np.isnan(neutral_maf1) & ~np.isnan(neutral_maf2)
        n1_neutral = neutral_maf1[keep_neut]
        n2_neutral = neutral_maf2[keep_neut]
        fst_neutral = np.mean(fst(n1_neutral, n2_neutral, n1=n1, n2=n2))
        Va_term = np.sum(beta ** 2 * 2 * maf1 * maf2)
        D_Qx = Va_term * fst_neutral
    else:
        fst_values = fst(maf1, maf2, n1=n1, n2=n2)
        D_Qx = np.sum((beta ** 2) * (2 * maf1 * maf2) * fst_values)
    Qx_obs = N_Qx_obs / D_Qx
    Fst_obs = np.mean(fst_values) if neutral_maf1 is None else fst_neutral
    N_Qx_perm = np.zeros(n_perm)
    for i in range(n_perm):
        beta_perm = np.random.permutation(beta)
        diff_perm = maf1 * beta_perm - maf2 * beta_perm
        N_Qx_perm[i] = np.sum(diff_perm) ** 2
    Qx_perm = N_Qx_perm / D_Qx
    empirical_p = np.mean(np.abs(Qx_perm) >= np.abs(Qx_obs))
    return {"Qx": Qx_obs, "Fst": Fst_obs, "p_value": empirical_p}

# Example run for the first test
test_result = compute_qx_cpu(maf1_list[0], maf2_list[0], beta_list[0], n_perm=1000, seed=seed)
print(f"QX (CPU) example: {test_result}")

QX (CPU) example: {'Qx': np.float64(0.5905964406662577), 'Fst': np.float64(0.01839083176232469), 'p_value': np.float64(0.785)}


## 4. QX Statistic: Sequential GPU (CuPy) Implementation

This function computes the QX statistic and empirical p-value using GPU acceleration (CuPy). It closely follows the R/CuPy logic for adaptive, memory-efficient computation.

In [5]:
def compute_qx_gpu_sequential(maf1, maf2, beta, n_perm=10000, seed=None, n1=None, n2=None, neutral_maf1=None, neutral_maf2=None, test_name=None, test_id=None):
    """
    Sequential GPU QX computation with adaptive OOM handling (CuPy).
    Returns dict with Qx, Fst, p_value, and exclusion info.
    """
    try:
        if seed is not None:
            cp.random.seed(int(seed))
        # Remove rows with any NA
        mask = ~np.isnan(maf1) & ~np.isnan(maf2) & ~np.isnan(beta)
        maf1 = maf1[mask]
        maf2 = maf2[mask]
        beta = beta[mask]
        n = len(beta)
        if n == 0:
            return {"Qx": np.nan, "Fst": np.nan, "p_value": np.nan, "excluded": False, "error": "No valid SNPs after NA removal"}
        # Transfer to GPU
        maf1_gpu = cp.asarray(maf1, dtype=cp.float64)
        maf2_gpu = cp.asarray(maf2, dtype=cp.float64)
        beta_gpu = cp.asarray(beta, dtype=cp.float64)
        # Observed Qx
        diff_obs = maf1_gpu * beta_gpu - maf2_gpu * beta_gpu
        N_Qx_obs = cp.sum(diff_obs) ** 2
        # Denominator
        if neutral_maf1 is not None and neutral_maf2 is not None:
            neut_mask = ~np.isnan(neutral_maf1) & ~np.isnan(neutral_maf2)
            neu1 = cp.asarray(neutral_maf1[neut_mask], dtype=cp.float64)
            neu2 = cp.asarray(neutral_maf2[neut_mask], dtype=cp.float64)
            fst_num = (maf1_gpu - maf2_gpu) ** 2
            fst_denom = neu1 * (1 - neu2) + neu2 * (1 - neu1) + 1e-12
            fst_neutral_global = cp.mean(fst_num / fst_denom)
            Va_term = cp.sum(beta_gpu ** 2 * 2 * maf1_gpu * maf2_gpu)
            D_Qx = Va_term * fst_neutral_global
            Fst_obs = float(fst_neutral_global.get())
        else:
            fst_num = (maf1_gpu - maf2_gpu) ** 2
            fst_denom = maf1_gpu * (1 - maf2_gpu) + maf2_gpu * (1 - maf1_gpu) + 1e-12
            fst_vals = cp.where(fst_denom > 0, fst_num / fst_denom, 0)
            D_Qx = cp.sum((beta_gpu ** 2) * (2 * maf1_gpu * maf2_gpu) * fst_vals)
            Fst_obs = float(cp.mean(fst_vals).get())
        Qx_obs = N_Qx_obs / D_Qx
        # Permutation test (all on GPU)
        n_gpu = int(n)
        n_perm_gpu = int(n_perm)
        random_matrix = cp.random.rand(n_gpu, n_perm_gpu).astype(cp.float32)
        perm_indices_all = cp.argsort(random_matrix, axis=0)
        beta_perm_all = beta_gpu[perm_indices_all]
        maf1_exp = cp.expand_dims(maf1_gpu, axis=1)
        maf2_exp = cp.expand_dims(maf2_gpu, axis=1)
        maf1_beta_perm = maf1_exp * beta_perm_all
        maf2_beta_perm = maf2_exp * beta_perm_all
        diff_perm_all = maf1_beta_perm - maf2_beta_perm
        diff_sum_perm = cp.sum(diff_perm_all, axis=0)
        N_Qx_perm = diff_sum_perm ** 2
        Qx_perm_all = N_Qx_perm / D_Qx
        Qx_perm_abs = cp.abs(Qx_perm_all)
        Qx_obs_abs = cp.abs(Qx_obs)
        exceeds = Qx_perm_abs >= Qx_obs_abs
        p_value_gpu = cp.mean(exceeds.astype(cp.float64))
        Qx_final = float(Qx_obs.get())
        Fst_final = Fst_obs
        p_value_final = float(p_value_gpu.get())
        # Aggressive GPU cleanup
        del maf1_gpu, maf2_gpu, beta_gpu, Qx_perm_all, fst_num, fst_denom, D_Qx, N_Qx_perm, Qx_obs_abs, diff_sum_perm, exceeds
        cp._default_memory_pool.free_bytes()
        cp.get_default_memory_pool().free_bytes()
        return {
            "Qx": Qx_final,
            "Fst": Fst_final,
            "p_value": p_value_final,
            "excluded": False,
            "test_name": test_name,
            "test_id": test_id,
            "n_snps": n
        }
    except Exception as e:
        return {
            "Qx": np.nan,
            "Fst": np.nan,
            "p_value": np.nan,
            "excluded": True,
            "error": str(e),
            "test_name": test_name,
            "test_id": test_id,
            "n_snps": len(beta) if 'beta' in locals() else None
        }

# Example run for the first test
test_result_gpu = compute_qx_gpu_sequential(maf1_list[0], maf2_list[0], beta_list[0], n_perm=1000, seed=seed)
print(f"QX (GPU) example: {test_result_gpu}")

QX (GPU) example: {'Qx': 0.5905964406662579, 'Fst': 0.018390831762324693, 'p_value': 0.757, 'excluded': False, 'test_name': None, 'test_id': None, 'n_snps': 487}


## 5. QX Statistic: Batch GPU (CuPy) Implementation

This function computes the QX statistic for multiple tests in parallel using GPU acceleration (CuPy), maximizing speed for large, uniform datasets. It processes tests in batches, each with its own perfectly-sized permutation matrix, and includes adaptive OOM handling.

In [ ]:
def compute_qx_gpu_batch(maf1_list, maf2_list, beta_list, n_perm=10000, seed=None, test_names=None, test_ids=None, batch_size=60, max_removal_attempts=50):
    """
    Batch GPU QX computation for multiple tests in parallel (CuPy).
    Returns a list of result dicts for each test.
    """
    n_tests = len(maf1_list)
    if test_names is None:
        test_names = [f"Test_{i+1}" for i in range(n_tests)]
    if test_ids is None:
        test_ids = [f"ID_{i+1}" for i in range(n_tests)]
    results = [None] * n_tests
    excluded_tests = []
    n_snps = [len(beta) for beta in beta_list]
    batch_starts = range(0, n_tests, batch_size)
    for batch_start in batch_starts:
        batch_end = min(batch_start + batch_size, n_tests)
        batch_indices = list(range(batch_start, batch_end))
        batch_removal_count = 0
        while batch_indices and batch_removal_count < max_removal_attempts:
            try:
                if seed is not None:
                    cp.random.seed(int(seed + batch_start))
                # Prepare batch data
                maf1_batch = [maf1_list[i] for i in batch_indices]
                maf2_batch = [maf2_list[i] for i in batch_indices]
                beta_batch = [beta_list[i] for i in batch_indices]
                n_snps_batch = [len(b) for b in beta_batch]
                # Generate permutation matrices for each test
                perm_matrices_gpu = [cp.argsort(cp.random.rand(n, n_perm).astype(cp.float32), axis=0) for n in n_snps_batch]
                maf1_gpu = [cp.asarray(m, dtype=cp.float64) for m in maf1_batch]
                maf2_gpu = [cp.asarray(m, dtype=cp.float64) for m in maf2_batch]
                beta_gpu = [cp.asarray(b, dtype=cp.float64) for b in beta_batch]
                Qx_obs_list, Fst_obs_list, p_value_list = [], [], []
                for i in range(len(batch_indices)):
                    # Observed Qx
                    diff_obs = maf1_gpu[i] * beta_gpu[i] - maf2_gpu[i] * beta_gpu[i]
                    N_Qx_obs = cp.sum(diff_obs) ** 2
                    fst_num = (maf1_gpu[i] - maf2_gpu[i]) ** 2
                    fst_denom = maf1_gpu[i] * (1 - maf2_gpu[i]) + maf2_gpu[i] * (1 - maf1_gpu[i]) + 1e-12
                    fst_vals = cp.where(fst_denom > 0, fst_num / fst_denom, 0)
                    D_Qx = cp.sum((beta_gpu[i] ** 2) * (2 * maf1_gpu[i] * maf2_gpu[i]) * fst_vals)
                    Fst_obs = float(cp.mean(fst_vals).get())
                    Qx_obs = N_Qx_obs / D_Qx
                    # Permutations
                    beta_perm = beta_gpu[i][perm_matrices_gpu[i]]
                    maf1_exp = maf1_gpu[i][:, None]
                    maf2_exp = maf2_gpu[i][:, None]
                    maf1_beta_perm = maf1_exp * beta_perm
                    maf2_beta_perm = maf2_exp * beta_perm
                    diff_perm = maf1_beta_perm - maf2_beta_perm
                    diff_sum_perm = cp.sum(diff_perm, axis=0)
                    N_Qx_perm = diff_sum_perm ** 2
                    Qx_perm = N_Qx_perm / D_Qx
                    p_value = float(cp.mean((cp.abs(Qx_perm) >= cp.abs(Qx_obs)).astype(cp.float64)).get())
                    Qx_obs_list.append(float(Qx_obs.get()))
                    Fst_obs_list.append(Fst_obs)
                    p_value_list.append(p_value)
                # Store results
                for idx, i in enumerate(batch_indices):
                    results[i] = {
                        "Qx": Qx_obs_list[idx],
                        "Fst": Fst_obs_list[idx],
                        "p_value": p_value_list[idx],
                        "test_name": test_names[i],
                        "test_id": test_ids[i],
                        "n_snps": n_snps[i],
                        "excluded": False
                    }
                break  # Batch succeeded
            except Exception as e:
                # OOM or other error: remove largest test and retry
                batch_removal_count += 1
                if len(batch_indices) == 1:
                    excluded_tests.append({
                        "index": batch_indices[0],
                        "test_name": test_names[batch_indices[0]],
                        "test_id": test_ids[batch_indices[0]],
                        "n_snps": n_snps[batch_indices[0]],
                        "error": str(e)
                    })
                    break
                largest_idx = batch_indices[np.argmax([n_snps[i] for i in batch_indices])]
                excluded_tests.append({
                    "index": largest_idx,
                    "test_name": test_names[largest_idx],
                    "test_id": test_ids[largest_idx],
                    "n_snps": n_snps[largest_idx],
                    "error": str(e)
                })
                batch_indices.remove(largest_idx)
    return {
        "results": results,
        "excluded": excluded_tests
    }

# Example run for the first 10 tests
batch_result = compute_qx_gpu_batch(maf1_list[:10], maf2_list[:10], beta_list[:10], n_perm=1000, seed=seed, test_names=test_names[:10], test_ids=test_ids[:10], batch_size=5)
for i, res in enumerate(batch_result['results']):
    print(f"Batch Test {i+1}: Qx={res['Qx']:.4f}, Fst={res['Fst']:.4f}, p={res['p_value']:.4g}")

In [ ]:
import random

def generate_simulated_data(n_snps=500, pop1_mean_maf=0.3, pop2_mean_maf=0.25, effect_sd=0.05, na_rate=0.02):
    """
    Generate simulated GWAS-like data for QX test.
    Returns dict with maf1, maf2, betas, test_name, test_id.
    """
    trait_names = [
        "Standing_height", "Body_mass_index", "Skin_colour", "Eye_colour",
        "Hair_colour", "Mean_corpuscular_volume", "Red_blood_cell_count",
        "Platelet_count", "White_blood_cell_count", "Hemoglobin_concentration",
        "Cholesterol_total", "HDL_cholesterol", "LDL_cholesterol", "Triglycerides",
        "Glucose", "C-reactive_protein", "Vitamin_D", "Bone_mineral_density"
    ]
    categories = ["continuous", "biomarkers", "categorical", "physical_measures"]
    test_name = random.choice(trait_names)
    category = random.choice(categories)
    trait_number = random.randint(1000, 9999)
    sex_category = random.choice(["both_sexes", "male", "female"])
    test_id = f"{category}-{trait_number}-{sex_category}"
    betas = np.random.normal(0, effect_sd, n_snps)
    maf1 = np.random.beta(2, 5, n_snps)
    maf1 = np.clip(maf1, 0.01, 0.5)
    maf2 = maf1 + np.random.normal(pop2_mean_maf - pop1_mean_maf, 0.05, n_snps)
    maf2 = np.clip(maf2, 0.01, 0.5)
    if na_rate > 0:
        n_missing = int(np.ceil(n_snps * na_rate))
        na_indices_maf1 = np.random.choice(n_snps, n_missing, replace=False)
        maf1[na_indices_maf1] = np.nan
        na_indices_maf2 = np.random.choice(n_snps, n_missing, replace=False)
        maf2[na_indices_maf2] = np.nan

    return {
        "maf1": maf1,
        "maf2": maf2,
        "betas": betas,
        "test_name": test_name,
        "test_id": test_id
    }

test
